# Teste de Extração: IBAMA Embargos
Baixa o Shapefile de embargos do IBAMA e converte para tabular.
**Requisito:** `pip install geopandas requests`

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
import tempfile
import os
from pathlib import Path

URL = "https://pamgia.ibama.gov.br/geoservicos/arquivos/adm_embargo_ibama_a.shp.zip"
OUTPUT = Path("data/bronze/ibama_embargos")
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Iniciando download (pode demorar)...")
try:
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
        r = requests.get(URL, stream=True, verify=False, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        for chunk in r.iter_content(chunk_size=8192): tmp.write(chunk)
        tmp_path = tmp.name

    print("Lendo Shapefile...")
    gdf = gpd.read_file(f"zip://{tmp_path}")
    df_tabular = pd.DataFrame(gdf.drop(columns=['geometry']))
    
    df_tabular.to_parquet(OUTPUT / "embargos.parquet")
    print(f"Sucesso! {len(df_tabular)} registros salvos.")
    os.remove(tmp_path)
except Exception as e:
    print(f"Falha: {e}")